In [ ]:
import string

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mxlpy import Simulator, scan
from mxlpy.parallel import parallelise

from mxlmodels import get_zhu_2009

## Figure 2

In [ ]:
m_perturbed = get_zhu_2009()
m_perturbed.remove_variable("RuBP")
m_perturbed.add_parameter("RuBP", 0.5)

res_phase1 = Simulator(model=get_zhu_2009()).simulate(650).get_result().unwrap_or_err()
res_phase2 = Simulator(model=m_perturbed).update_variables(res_phase1.get_new_y0()).simulate(50).get_result().unwrap_or_err()
res_phase3 = Simulator(model=get_zhu_2009()).update_variables(res_phase2.get_new_y0()).simulate(300).get_result().unwrap_or_err()

res_phase1 = res_phase1.get_variables()
res_phase2 = res_phase2.get_variables().iloc[1:]
res_phase2.index += 650
res_phase3 = res_phase3.get_variables().iloc[1:]
res_phase3.index += 650 + 50

res_fig2 = pd.concat([res_phase1, res_phase2, res_phase3])

In [ ]:
fig, axs = plt.subplots(3, 2, sharex=True, figsize=(8, 8))

style_dict = {
    "RuBP": {
        "ax": axs[0, 0],
        "ylim": (0, 4),
        "yticks": np.linspace(0, 4, 5),
        "ylabel": "Metabolite Concentration (mM)",
        "letter": "A",
        "yaxis_pos": "left"
    },
    "PGA": {
        "ax": axs[0, 1],
        "ylim": (-1, 3),
        "yticks": np.linspace(0, 3, 4),
        "ylabel": "Metabolite Concentration (mM)",
        "letter": "B",
        "yaxis_pos": "right"
    },
    "DPGA": {
        "ax": axs[1, 0],
        "ylim": (0, 3),
        "yticks": np.linspace(0, 3, 4),
        "ylabel": "Metabolite Concentration (mM)",
        "letter": "C",
        "yaxis_pos": "left"
    },
    "GAP": {
        "ax": axs[1, 1],
        "ylim": (0, 8),
        "yticks": np.linspace(0, 8, 5),
        "ylabel": "Metabolite Concentration (mM)",
        "letter": "D",
        "yaxis_pos": "right"
    },
    "Ru5P": {
        "ax": axs[2, 0],
        "ylim": (0.00, 0.10),
        "yticks": np.linspace(0.00, 0.09, 4),
        "ylabel": "Metabolite Concentration (mM)",
        "letter": "E",
        "yaxis_pos": "left"
    },
    "A": {
        "ax": axs[2, 1],
        "ylim": (35, 60),
        "yticks": np.linspace(35, 60, 6),
        "ylabel": "A (µmol m⁻² s⁻¹)",
        "letter": "F",
        "yaxis_pos": "right"
    },
}

for var, style in style_dict.items():
    ax = style["ax"]
    
    ax.plot(res_fig2[var], color="black")
    ax.set_ylim(style["ylim"])
    ax.set_yticks(style["yticks"])
    ax.set_ylabel(style["ylabel"])
    ax.text(0.05, 0.9, style["letter"], transform=ax.transAxes)
    ax.text(0.1, 0.9, var, transform=ax.transAxes)
    
    if style["yaxis_pos"] == "right":
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position("right")
        
    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        bottom=True,
        top=True,
        left=True,
        right=True
    )
        
    ax.set_xlim(0, 1000)
    ax.set_xticks(np.linspace(0, 1000, 6))
    if var in ["Ru5P", "A"]:
        ax.set_xlabel("Time (s)")


plt.tight_layout()
plt.show()

## Figure 3

In [ ]:
def steady_state_scan_fig3(param):
    res = scan.steady_state(
        model=get_zhu_2009(),
        to_scan=pd.DataFrame({param: get_zhu_2009().get_parameter_values()[param] * (np.arange(50, 350, step=50) / 100)})
    )
    
    return res.variables

params_to_change = ["V1_max", "V3_max", "V4_max", "V13_max"]

res_fig3 = dict(
    parallelise(
        steady_state_scan_fig3,
        [(param, param) for param in params_to_change]
    )
)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

for param, (i, ax) in zip(res_fig3.keys(), enumerate(axs.flat)):
    res = res_fig3[param].copy()
    res.index /= get_zhu_2009().get_parameter_values()[param] / 100
    ax.plot(res["RuBP"], marker="o", ls="none", color="black", label="RuBP")
    ax.plot(res["PGA"], marker="o", ls="none", color="black", markerfacecolor="white", label="PGA")
    ax.plot(res["DPGA"], marker="v", ls="none", color="black", label="DPGA")
    ax.plot(res["Ru5P"], marker="v", ls="none", color="black", markerfacecolor="white", label="Ru5P")

    ax.text(0.05, 0.95, string.ascii_uppercase[i], transform=ax.transAxes, va="top")
    ax.text(0.1, 0.95, param, transform=ax.transAxes, va="top")
    ax.set_ylabel("Concentration (mM)")
    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        bottom=True,
        top=True,
        left=True,
        right=True
    )
    
axs[0, 0].set_ylim(0.0, 5.7)
axs[0, 0].set_yticks(np.linspace(0.0, 5.6, 5))
axs[0, 0].legend()

axs[0, 1].set_ylim(0.0, 1.2)
axs[0, 1].set_yticks(np.linspace(0.0, 1.2, 5))
axs[0, 1].yaxis.tick_right()
axs[0, 1].yaxis.set_label_position("right")

axs[1, 0].set_ylim(0, 8)
axs[1, 0].set_yticks(np.linspace(0, 6, 4))
axs[1, 0].set_xlabel("Enzyme activity (Percentage of default %)")

axs[1, 1].set_ylim(0.0, 1.0)
axs[1, 1].set_yticks(np.linspace(0.0, 0.9, 4))
axs[1, 1].yaxis.tick_right()
axs[1, 1].yaxis.set_label_position("right")
axs[1, 1].set_xlabel("Enzyme activity (Percentage of default %)")

plt.tight_layout()
plt.show()

## Figure 4 

In [ ]:
params_to_change = ["V2_max", "V5_max"]

res_fig4 = dict(
    parallelise(
        steady_state_scan_fig3,
        [(param, param) for param in params_to_change]
    )
)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(10, 3))

for param, (i, ax) in zip(res_fig4.keys(), enumerate(axs.flat)):
    res = res_fig4[param].copy()
    res.index /= get_zhu_2009().get_parameter_values()[param] / 100
    ax.plot(res["RuBP"], marker="o", ls="none", color="black", label="RuBP")
    ax.plot(res["PGA"], marker="o", ls="none", color="black", markerfacecolor="white", label="PGA")
    ax.plot(res["DPGA"], marker="v", ls="none", color="black", label="DPGA")
    ax.plot(res["Ru5P"], marker="v", ls="none", color="black", markerfacecolor="white", label="Ru5P")

    ax.text(0.05, 0.95, string.ascii_uppercase[i], transform=ax.transAxes, va="top")
    ax.text(0.1, 0.95, param, transform=ax.transAxes, va="top")
    ax.set_ylabel("Concentration (mM)")
    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        bottom=True,
        top=True,
        left=True,
        right=True
    )
    ax.set_ylim(0, 1.0)
    ax.set_yticks(np.linspace(0, 0.9, 4))
    ax.set_xlabel("Enzyme activity (Percentage of default %)")
    ax.set_xlim(40, 360)

axs[0].legend()
axs[1].yaxis.tick_right()
axs[1].yaxis.set_label_position("right")


plt.tight_layout()
plt.show()